# Target 2 — Load/Demand (item_count_sum) Forecast — 7-Day
**AstroLogix Logistics Forecasting System**

## What are we doing?
We forecast the **item_count** column sum per region × day from `orders.parquet` as a load/demand proxy.

## Why this approach?
Per the project lead's note: *"Target 2 is basically order count × average weight"* — so we solve it with the same pipeline, just a different target.

## Pipeline Summary
```
orders.parquet (timestamp → hour/day/month/year extraction)
    ↓
Hourly pattern → daily aggregate features
    ↓
Holidays + Weather data
    ↓
Full Feature Engineering (leak-free, shift(1))
    ↓
Time-based train/test split (NO shuffle!)
    ↓
6 ML models → comparison → best selected
    ↓
Retrain on full history → recursive 7-day forecast
```

In [49]:
import pandas as pd
import numpy as np
import warnings
import random
import joblib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    train_test_split,
    TimeSeriesSplit,
    KFold,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor
)

from sklearn.svm import SVR

from sklearn.neighbors import KNeighborsRegressor

from sklearn.tree import DecisionTreeRegressor
import bisect
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.model_selection import TimeSeriesSplit
SEED = 42

np.random.seed(SEED)
random.seed(SEED)

plt.style.use("seaborn-v0_8-whitegrid")

print("✅ All libraries loaded successfully.")

✅ All libraries loaded successfully.


## 1. Data Loading — Load Parquet Files

In [50]:
orders   = pd.read_parquet('orders.parquet')
holidays = pd.read_parquet('holidays.parquet')
weather  = pd.read_parquet('weather.parquet')

# created_at → datetime (strip timezone offset)
orders['created_at'] = pd.to_datetime(
    orders['created_at'].astype(str).str.replace(r'\+00:00$', '', regex=True)
)

print(f"Orders   : {orders.shape}  |  columns: {list(orders.columns)}")
print(f"Holidays : {holidays.shape}")
print(f"Weather  : {weather.shape}")
print(f"Date range: {orders['created_at'].min()} -> {orders['created_at'].max()}")
print(f"Regions  : {sorted(orders['region'].unique())}")


Orders   : (110000, 8)  |  columns: ['order_id', 'region', 'item_count', 'created_at', 'delivery_type', 'shipment_id', 'fulfilling_warehouse_id', 'destination_store_id']
Holidays : (135, 3)
Weather  : (23570, 5)
Date range: 2020-01-01 00:49:03 -> 2026-06-12 23:06:16
Regions  : ['Absheron', 'Ganja', 'Kalbajar', 'Khachmaz', 'Khankendi', 'Lankaran', 'Nakhchivan', 'Qazakh', 'Sheki', 'Yevlakh']


## 2. Timestamp Decomposition — Hour, Day, Month, Year Features

The `created_at` column contains both date and time information.  
We extract **hour**-based behaviour features from each order, then aggregate them as daily columns.

**Why hour features?**  
- Orders arriving at fixed hours (e.g. lunch, evening) reveal a region's logistics profile  
- The share of night orders signals express delivery demand  
- "Order hour standard deviation" shows how spread out demand is throughout the day  

⚠️ These features will be converted to **daily sums**, then lagged with **shift(1)** — no leakage.

In [51]:
# ── Extract time components from timestamp ────────────────────────────────────
orders['date']   = orders['created_at'].dt.normalize()
orders['hour']   = orders['created_at'].dt.hour
orders['minute'] = orders['created_at'].dt.minute

# Intra-day time-of-day buckets
orders['is_early_morning']  = ((orders['hour'] >= 5)  & (orders['hour'] < 9)).astype(int)
orders['is_morning']        = ((orders['hour'] >= 9)  & (orders['hour'] < 12)).astype(int)
orders['is_afternoon']      = ((orders['hour'] >= 12) & (orders['hour'] < 17)).astype(int)
orders['is_evening']        = ((orders['hour'] >= 17) & (orders['hour'] < 21)).astype(int)
orders['is_night']          = ((orders['hour'] >= 21) | (orders['hour'] < 5)).astype(int)
orders['is_business_hours'] = ((orders['hour'] >= 9)  & (orders['hour'] < 18)).astype(int)
orders['is_lunch_rush']     = ((orders['hour'] >= 11) & (orders['hour'] < 14)).astype(int)
orders['is_express']        = (orders['delivery_type'] == 'express').astype(int)

print("Hour features created.")
orders[['created_at', 'hour', 'is_morning', 'is_evening', 'is_night']].head(5)


Hour features created.


,created_at,hour,is_morning,is_evening,is_night
0,2026-06-07 06:33:53,6,0,0,0
1,2022-11-01 03:41:42,3,0,0,1
2,2021-06-19 09:03:08,9,1,0,0
3,2023-07-27 09:38:15,9,1,0,0
4,2025-09-12 00:41:11,0,0,0,1


## 3. Daily Aggregation — Region × Date Grid

For each (region, day) pair:
- **Main target**: `item_count_sum` — total item count across all orders from that region that day
- **Auxiliary features**: hourly patterns, express ratio, average items per order

In [52]:
# ── Daily aggregation ─────────────────────────────────────────────────────────
daily = orders.groupby(['date', 'region']).agg(
    order_count         = ('order_id',          'count'),
    item_count_sum      = ('item_count',         'sum'),
    avg_items_per_order = ('item_count',         'mean'),
    express_ratio       = ('is_express',         'mean'),
    # Hour pattern features
    avg_order_hour      = ('hour',               'mean'),
    order_hour_std      = ('hour',               'std'),
    early_morning_ratio = ('is_early_morning',   'mean'),
    morning_ratio       = ('is_morning',         'mean'),
    afternoon_ratio     = ('is_afternoon',       'mean'),
    evening_ratio       = ('is_evening',         'mean'),
    night_ratio         = ('is_night',           'mean'),
    business_hours_ratio= ('is_business_hours',  'mean'),
    lunch_rush_ratio    = ('is_lunch_rush',      'mean'),
    peak_hour           = ('hour', lambda x: int(x.value_counts().idxmax())),
).reset_index()

# ── Full region × date grid (missing days filled with 0) ─────────────────────
regions    = sorted(orders['region'].unique())
date_range = pd.date_range(daily['date'].min(), daily['date'].max(), freq='D')
full_idx   = pd.MultiIndex.from_product([date_range, regions], names=['date', 'region'])
full       = pd.DataFrame(index=full_idx).reset_index()

# Default values for missing days
fill_vals = {
    'order_count': 0, 'item_count_sum': 0,
    'avg_items_per_order': 0, 'express_ratio': 0,
    'avg_order_hour': 12, 'order_hour_std': 0,
    'early_morning_ratio': 0, 'morning_ratio': 0,
    'afternoon_ratio': 0, 'evening_ratio': 0,
    'night_ratio': 0, 'business_hours_ratio': 0,
    'lunch_rush_ratio': 0, 'peak_hour': 12,
}

daily = full.merge(daily, on=['date', 'region'], how='left').fillna(fill_vals)

print(f"Daily grid : {daily.shape}")
print(f"Date range : {daily['date'].min().date()} -> {daily['date'].max().date()}")
print(f"Regions    : {regions}")
daily.tail()


Daily grid : (23550, 16)
Date range : 2020-01-01 -> 2026-06-12
Regions    : ['Absheron', 'Ganja', 'Kalbajar', 'Khachmaz', 'Khankendi', 'Lankaran', 'Nakhchivan', 'Qazakh', 'Sheki', 'Yevlakh']


,date,region,order_count,item_count_sum,avg_items_per_order,express_ratio,avg_order_hour,order_hour_std,early_morning_ratio,morning_ratio,afternoon_ratio,evening_ratio,night_ratio,business_hours_ratio,lunch_rush_ratio,peak_hour
23545,2026-06-12,Lankaran,1.0,4.0,4.000000,1.000000,12.000000,0.000000,0.000000,0.0,1.0,0.00,0.000000,1.000000,1.000000,12.0
23546,2026-06-12,Nakhchivan,12.0,64.0,5.333333,0.416667,10.166667,6.206058,0.083333,0.5,0.0,0.25,0.166667,0.583333,0.083333,9.0
23547,2026-06-12,Qazakh,1.0,7.0,7.000000,0.000000,22.000000,0.000000,0.000000,0.0,0.0,0.00,1.000000,0.000000,0.000000,22.0
23548,2026-06-12,Sheki,0.0,0.0,0.000000,0.000000,12.000000,0.000000,0.000000,0.0,0.0,0.00,0.000000,0.000000,0.000000,12.0
23549,2026-06-12,Yevlakh,0.0,0.0,0.000000,0.000000,12.000000,0.000000,0.000000,0.0,0.0,0.00,0.000000,0.000000,0.000000,12.0


## 4. Holiday and Weather Features

In [53]:
# ── Holiday features ──────────────────────────────────────────────────────────
holidays['date'] = pd.to_datetime(holidays['date'])
hol_sorted = sorted(holidays['date'].tolist())
hol_set    = set(hol_sorted)

def days_to_holiday(d):
    idx  = bisect.bisect_left(hol_sorted, d)
    best = 9999
    if idx < len(hol_sorted):
        best = min(best, (hol_sorted[idx] - d).days)
    if idx > 0:
        best = min(best, (d - hol_sorted[idx - 1]).days)
    return best

def days_to_next_holiday(d):
    idx = bisect.bisect_right(hol_sorted, d)
    return (hol_sorted[idx] - d).days if idx < len(hol_sorted) else 999

def days_from_last_holiday(d):
    idx = bisect.bisect_left(hol_sorted, d)
    return (d - hol_sorted[idx - 1]).days if idx > 0 else 999

daily['is_holiday']            = daily['date'].isin(hol_set).astype(int)
daily['days_to_holiday']       = daily['date'].apply(days_to_holiday)
daily['days_to_next_holiday']  = daily['date'].apply(days_to_next_holiday)
daily['days_from_last_holiday']= daily['date'].apply(days_from_last_holiday)
daily['is_holiday_eve']        = daily['date'].apply(lambda d: int((d + pd.Timedelta(days=1)) in hol_set))
daily['is_holiday_after']      = daily['date'].apply(lambda d: int((d - pd.Timedelta(days=1)) in hol_set))
daily['is_long_weekend']       = (
    (daily['date'].dt.dayofweek.isin([4, 5, 6])) &
    ((daily['is_holiday'] == 1) | (daily['is_holiday_eve'] == 1) | (daily['is_holiday_after'] == 1))
).astype(int)
daily['holiday_week'] = daily['days_to_holiday'].apply(lambda x: int(x <= 3))

# ── Weather features ──────────────────────────────────────────────────────────
weather['date'] = pd.to_datetime(
    weather['timestamp'].astype(str).str.replace(r'\+00:00$', '', regex=True)
).dt.normalize()
# Keep only regions present in orders
weather = weather[weather['region'].isin(regions)]

wagg = weather.groupby(['date', 'region']).agg(
    temperature = ('temperature', 'mean'),
    rainfall    = ('rainfall',    'sum'),
    wind_speed  = ('wind_speed',  'mean'),
    temp_min    = ('temperature', 'min'),
    temp_max    = ('temperature', 'max'),
).reset_index()

wagg['temp_range']  = wagg['temp_max'] - wagg['temp_min']
wagg['feels_cold']  = (wagg['temperature'] < 5).astype(int)
wagg['feels_hot']   = (wagg['temperature'] > 35).astype(int)
wagg['heavy_rain']  = (wagg['rainfall'] > 10).astype(int)
wagg['strong_wind'] = (wagg['wind_speed'] > 40).astype(int)

daily = daily.merge(wagg, on=['date', 'region'], how='left')

# Fill missing weather with region+month median
weather_cols = ['temperature', 'rainfall', 'wind_speed', 'temp_min', 'temp_max',
                'temp_range', 'feels_cold', 'feels_hot', 'heavy_rain', 'strong_wind']
daily['month_tmp'] = daily['date'].dt.month
for col in weather_cols:
    med = daily.groupby(['region', 'month_tmp'])[col].transform('median')
    daily[col] = daily[col].fillna(med)
    daily[col] = daily.groupby('region')[col].transform(lambda x: x.ffill().bfill())
daily.drop(columns=['month_tmp'], inplace=True)

print(f"Null values: {daily.isna().sum().sum()}")
daily.head()


Null values: 0


,date,region,order_count,item_count_sum,avg_items_per_order,express_ratio,avg_order_hour,order_hour_std,early_morning_ratio,morning_ratio,...,temperature,rainfall,wind_speed,temp_min,temp_max,temp_range,feels_cold,feels_hot,heavy_rain,strong_wind
0,2020-01-01,Absheron,32.0,246.0,7.687500,0.281250,10.500000,6.834448,0.250000,0.125000,...,8.450,0.0,32.343952,8.450,8.450,0.0,0,0,0,0
1,2020-01-01,Ganja,17.0,72.0,4.235294,0.411765,9.294118,5.096135,0.294118,0.352941,...,9.100,0.1,12.429127,9.100,9.100,0.0,0,0,0,0
2,2020-01-01,Kalbajar,0.0,0.0,0.000000,0.000000,12.000000,0.000000,0.000000,0.000000,...,4.602,3.4,8.854829,4.602,4.602,0.0,1,0,0,0
3,2020-01-01,Khachmaz,5.0,31.0,6.200000,0.400000,14.400000,8.234076,0.000000,0.000000,...,6.339,1.3,6.989935,6.339,6.339,0.0,0,0,0,0
4,2020-01-01,Khankendi,1.0,1.0,1.000000,1.000000,12.000000,0.000000,0.000000,0.000000,...,6.322,4.0,6.696387,6.322,6.322,0.0,0,0,0,0


## 5. Comprehensive Feature Engineering (Leak-Free)

### Leakage Prevention Rule
> All lag and rolling features are computed with `shift(1)`.  
> The model never sees **today's** value — only **yesterday and earlier**.

### Feature Categories Created
| # | Category | Count | Example |
|---|---|---|---|
| 1 | **Date** (year, month, day, hour-based) | ~25 | `year`, `month`, `day` |
| 2 | **Cyclic encoding** | 8 | `dow_sin`, `month_sin`, `week_sin` |
| 3 | **Holiday/vacation** | 8 | `is_holiday_eve`, `days_to_next_holiday` |
| 4 | **Lag** (lookback) | 16 | `lag_1`, `lag_7`, `lag_365` |
| 5 | **Rolling** (moving statistics) | 20 | `roll_mean_7`, `roll_std_28`, `roll_max_14` |
| 6 | **Momentum** | 8 | `lag_diff_1_7`, `yoy_ratio`, `wow_ratio` |
| 7 | **Hour pattern lags** | 10 | `morning_ratio_lag1`, `express_ratio_roll7` |
| 8 | **Region statistics** | 10 | `region_dow_mean`, `region_season_mean` |
| 9 | **Weather** | 10 | `temperature`, `heavy_rain` |
| 10 | **Market share** | 3 | `region_market_share`, `total_daily_orders_lag1` |

In [54]:
TARGET = 'item_count_sum'

# Hour pattern feature columns — also lagged to prevent leakage
HOUR_PATTERN_COLS = [
    'avg_order_hour', 'order_hour_std',
    'early_morning_ratio', 'morning_ratio', 'afternoon_ratio',
    'evening_ratio', 'night_ratio', 'business_hours_ratio',
    'lunch_rush_ratio', 'avg_items_per_order', 'express_ratio',
    'peak_hour',
]

def add_features(df, target_col):
    df = df.copy().sort_values(['region', 'date']).reset_index(drop=True)

    # ── 1. Date features ─────────────────────────────────────────────────────
    df['year']              = df['date'].dt.year
    df['month']             = df['date'].dt.month
    df['day']               = df['date'].dt.day
    df['dayofweek']         = df['date'].dt.dayofweek
    df['dayofyear']         = df['date'].dt.dayofyear
    df['quarter']           = df['date'].dt.quarter
    df['weekofyear']        = df['date'].dt.isocalendar().week.astype(int)
    df['days_in_month']     = df['date'].dt.days_in_month
    df['remaining_days_in_month'] = df['days_in_month'] - df['day']
    df['is_weekend']        = (df['dayofweek'] >= 5).astype(int)
    df['is_weekday']        = (df['dayofweek'] < 5).astype(int)
    df['is_month_start']    = df['date'].dt.is_month_start.astype(int)
    df['is_month_end']      = df['date'].dt.is_month_end.astype(int)
    df['is_quarter_start']  = df['date'].dt.is_quarter_start.astype(int)
    df['is_quarter_end']    = df['date'].dt.is_quarter_end.astype(int)
    df['is_first_week']     = (df['day'] <= 7).astype(int)
    df['is_last_week']      = (df['remaining_days_in_month'] <= 7).astype(int)
    df['is_mid_month']      = ((df['day'] >= 13) & (df['day'] <= 17)).astype(int)

    df['season'] = df['month'].map({12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
                                     6:2, 7:2, 8:2, 9:3, 10:3, 11:3})
    df['is_summer'] = (df['season'] == 2).astype(int)
    df['is_winter'] = (df['season'] == 0).astype(int)
    df['trend'] = (df['date'] - df['date'].min()).dt.days

    # ── 2. Cyclic encoding ───────────────────────────────────────────────────
    df['dow_sin']   = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['dayofweek'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['week_sin']  = np.sin(2 * np.pi * df['weekofyear'] / 52)
    df['week_cos']  = np.cos(2 * np.pi * df['weekofyear'] / 52)
    df['doy_sin']   = np.sin(2 * np.pi * df['dayofyear'] / 366)
    df['doy_cos']   = np.cos(2 * np.pi * df['dayofyear'] / 366)

    # ── 3. Lag the hour pattern features (prevent leakage) ───────────────────
    for col in HOUR_PATTERN_COLS:
        if col in df.columns:
            grp = df.groupby('region')[col]
            df[f'{col}_lag1']   = grp.shift(1)
            df[f'{col}_roll7']  = grp.shift(1).groupby(df['region']).rolling(7, min_periods=1).mean().reset_index(drop=True)
            df[f'{col}_roll14'] = grp.shift(1).groupby(df['region']).rolling(14, min_periods=1).mean().reset_index(drop=True)

    # ── 4. Lag features ──────────────────────────────────────────────────────
    gcol = df.groupby('region')[target_col]

    for lag in [1, 2, 3, 7, 14, 21, 28, 364, 365, 366]:
        df[f'lag_{lag}'] = gcol.shift(lag)

    df['same_dow_last_week']  = df.groupby(['region', 'dayofweek'])[target_col].shift(1)
    df['same_dow_2weeks_ago'] = df.groupby(['region', 'dayofweek'])[target_col].shift(2)
    df['same_dow_3weeks_ago'] = df.groupby(['region', 'dayofweek'])[target_col].shift(3)
    df['same_dow_4weeks_ago'] = df.groupby(['region', 'dayofweek'])[target_col].shift(4)

    if 'order_count' in df.columns:
        oc = df.groupby('region')['order_count']
        for lag in [1, 7, 28]:
            df[f'order_count_lag{lag}'] = oc.shift(lag)
        df['order_count_roll7'] = oc.shift(1).groupby(df['region']).rolling(7, min_periods=1).mean().reset_index(drop=True)

    # ── 5. Rolling features ──────────────────────────────────────────────────
    shifted = gcol.shift(1)

    for win in [3, 7, 14, 28]:
        g = shifted.groupby(df['region'])
        df[f'roll_mean_{win}']   = g.rolling(win, min_periods=1).mean().reset_index(drop=True)
        df[f'roll_std_{win}']    = g.rolling(win, min_periods=1).std().reset_index(drop=True).fillna(0)
        df[f'roll_max_{win}']    = g.rolling(win, min_periods=1).max().reset_index(drop=True)
        df[f'roll_min_{win}']    = g.rolling(win, min_periods=1).min().reset_index(drop=True)
        df[f'roll_median_{win}'] = g.rolling(win, min_periods=1).median().reset_index(drop=True)

    df['roll_mean_90']  = shifted.groupby(df['region']).rolling(90,  min_periods=14).mean().reset_index(drop=True)
    df['roll_mean_365'] = shifted.groupby(df['region']).rolling(365, min_periods=30).mean().reset_index(drop=True)

    df['ewm_span7']  = shifted.groupby(df['region']).transform(lambda x: x.ewm(span=7,  adjust=False).mean())
    df['ewm_span14'] = shifted.groupby(df['region']).transform(lambda x: x.ewm(span=14, adjust=False).mean())
    df['ewm_span28'] = shifted.groupby(df['region']).transform(lambda x: x.ewm(span=28, adjust=False).mean())

    # ── 6. Momentum features ─────────────────────────────────────────────────
    df['lag_diff_1_7']   = df['lag_1']  - df['lag_7']
    df['lag_diff_7_14']  = df['lag_7']  - df['lag_14']
    df['lag_diff_7_28']  = df['lag_7']  - df['lag_28']
    df['lag_ratio_1_7']  = df['lag_1']  / (df['lag_7']  + 1)
    df['lag_ratio_7_28'] = df['lag_7']  / (df['lag_28'] + 1)
    df['yoy_ratio']      = df['lag_7']  / (df['lag_365'] + 1)
    df['wow_ratio']      = df['roll_mean_7'] / (df['roll_mean_28'] + 1)
    df['roll_range_7']   = df['roll_max_7']  - df['roll_min_7']
    df['roll_range_28']  = df['roll_max_28'] - df['roll_min_28']
    df['mom_growth']     = (df['roll_mean_7'] - df['roll_mean_28']) / (df['roll_mean_28'] + 1)

    # ── 7. Market Share Features ─────────────────────────────────────────────
    date_total = (
        df.groupby('date')[target_col]
          .sum()
          .rename('date_total')
          .reset_index()
    )
    df = df.merge(date_total, on='date', how='left')
    df['region_market_share'] = df[target_col] / (df['date_total'] + 1)
    date_total = date_total.sort_values('date')
    date_total['total_daily_lag1'] = date_total['date_total'].shift(1)
    df = df.merge(date_total[['date', 'total_daily_lag1']], on='date', how='left')
    df['market_share_lag1'] = df.groupby('region')['region_market_share'].shift(1)
    df['market_share_roll7'] = (
        df.groupby('region')['region_market_share']
          .shift(1)
          .groupby(df['region'])
          .rolling(7, min_periods=1)
          .mean()
          .reset_index(level=0, drop=True)
    )
    df.drop(columns=['date_total', 'region_market_share'], inplace=True, errors='ignore')

    # ── 8. Region encoding ───────────────────────────────────────────────────
    for col in ['region_mean', 'region_std', 'region_median', 'region_max', 'region_min',
                'region_dow_mean', 'region_month_mean', 'region_season_mean', 'region_week_mean']:
        df[col] = np.nan

    df['lag1_vs_region_mean']  = np.nan
    df['roll7_vs_region_mean'] = np.nan
    df['roll7_vs_dow_mean']    = np.nan

    df['region_cat'] = df['region'].astype('category').cat.codes

    return df


# ── Apply leak-free features (NO shuffle — time order preserved) ──────────────
feat = add_features(daily, TARGET)
feat = feat.sort_values(['region', 'date']).reset_index(drop=True)
feat = feat.dropna(subset=[c for c in feat.columns
                            if c not in ['region_mean', 'region_std', 'region_median',
                                         'region_max', 'region_min', 'region_dow_mean',
                                         'region_month_mean', 'region_season_mean',
                                         'region_week_mean', 'lag1_vs_region_mean',
                                         'roll7_vs_region_mean', 'roll7_vs_dow_mean']]).reset_index(drop=True)

print("Feature engineering complete (region stats deferred to post-split).")
print(f"Shape        : {feat.shape}")
print(f"Date range   : {feat['date'].min().date()} -> {feat['date'].max().date()}")
print(f"First 5 cols : {list(feat.columns[:5])}")
print(f"Last 5 cols  : {list(feat.columns[-5:])}")

Feature engineering complete (region stats deferred to post-split).
Shape        : (19890, 169)
Date range   : 2021-01-01 -> 2026-06-12
First 5 cols : ['date', 'region', 'order_count', 'item_count_sum', 'avg_items_per_order']
Last 5 cols  : ['region_week_mean', 'lag1_vs_region_mean', 'roll7_vs_region_mean', 'roll7_vs_dow_mean', 'region_cat']


## 6. Feature Summary

In [55]:
drop_cols  = ['date', 'region', 'order_count', 'item_count_sum']
# Also drop raw hour pattern columns (keep the lagged versions)
drop_cols += HOUR_PATTERN_COLS

feature_cols = [c for c in feat.columns if c not in drop_cols]

print(f"Feature count for model: {len(feature_cols)}")
print("\nFeatures:")
for i, f in enumerate(feature_cols):
    print(f"  {i+1:3d}. {f}")


Feature count for model: 153

Features:
    1. is_holiday
    2. days_to_holiday
    3. days_to_next_holiday
    4. days_from_last_holiday
    5. is_holiday_eve
    6. is_holiday_after
    7. is_long_weekend
    8. holiday_week
    9. temperature
   10. rainfall
   11. wind_speed
   12. temp_min
   13. temp_max
   14. temp_range
   15. feels_cold
   16. feels_hot
   17. heavy_rain
   18. strong_wind
   19. year
   20. month
   21. day
   22. dayofweek
   23. dayofyear
   24. quarter
   25. weekofyear
   26. days_in_month
   27. remaining_days_in_month
   28. is_weekend
   29. is_weekday
   30. is_month_start
   31. is_month_end
   32. is_quarter_start
   33. is_quarter_end
   34. is_first_week
   35. is_last_week
   36. is_mid_month
   37. season
   38. is_summer
   39. is_winter
   40. trend
   41. dow_sin
   42. dow_cos
   43. month_sin
   44. month_cos
   45. week_sin
   46. week_cos
   47. doy_sin
   48. doy_cos
   49. avg_order_hour_lag1
   50. avg_order_hour_roll7
   51. avg_orde

## 7. Train / Test Split (Time-Based)

**Why not random split?**  
Random splitting in a time series lets the future leak into the training set,  
producing perfect (but fake) test metrics.

✅ **Correct approach:** Keep all data in chronological order; hold out the last 20% as the test set.  
⚠️ `sample()` is NOT called.

In [56]:
# -----------------------------
# TRAIN / TEST SPLIT
# -----------------------------

X = feat[feature_cols]
y = feat[TARGET]

# Date-based split
split_date = feat['date'].quantile(0.80, interpolation='nearest')

train_mask = feat['date'] <= split_date
test_mask  = feat['date'] >  split_date

X_train = X.loc[train_mask].reset_index(drop=True)
X_test  = X.loc[test_mask].reset_index(drop=True)

y_train = y.loc[train_mask].reset_index(drop=True)
y_test  = y.loc[test_mask].reset_index(drop=True)

dates_test   = feat.loc[test_mask, 'date'].reset_index(drop=True)
regions_test = feat.loc[test_mask, 'region'].reset_index(drop=True)

# Log transform
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

# StandardScaler (for Ridge)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("="*60)
print("TRAIN / TEST SUMMARY")
print("="*60)
print("Train rows :", len(X_train))
print("Test rows  :", len(X_test))
print("Train target unique :", y_train.nunique())
print("Test target unique  :", y_test.nunique())
print("Train date range:")
print(feat.loc[train_mask, 'date'].min(), "->", feat.loc[train_mask, 'date'].max())
print()
print("Test date range:")
print(feat.loc[test_mask, 'date'].min(), "->", feat.loc[test_mask, 'date'].max())
print("="*60)


TRAIN / TEST SUMMARY
Train rows : 15920
Test rows  : 3970
Train target unique : 228
Test target unique  : 176
Train date range:
2021-01-01 00:00:00 -> 2025-05-11 00:00:00

Test date range:
2025-05-12 00:00:00 -> 2026-06-12 00:00:00


## 8. XGBoost — Default Parameters, then Hyperparameter Search


In [57]:
results     = {}
best_models = {}

def run_search(name, estimator, param_grid, Xtr, Xte,
               ytr=None, yte=None, log_transform=False):

    ytr = y_train if ytr is None else ytr
    yte = y_test  if yte is None else yte
    y_fit = np.log1p(ytr) if log_transform else ytr

    n_comb = max(1, int(np.prod([len(v) for v in param_grid.values()]))) if param_grid else 1
    print(f"\n{'='*62}")
    print(f"  Model        : {name}")
    print(f"  Combinations : {n_comb}   |   Folds: 10   |   Total fits: {n_comb*10}")
    print(f"{'='*62}")

    tscv = TimeSeriesSplit(n_splits=10)
    search = GridSearchCV(estimator, param_grid=param_grid,
                          scoring='r2', cv=tscv, n_jobs=-1, verbose=0)
    search.fit(Xtr, y_fit)
    best = search.best_estimator_

    raw_tr = best.predict(Xtr)
    raw_te = best.predict(Xte)
    pred_tr = np.expm1(raw_tr) if log_transform else raw_tr
    pred_te = np.expm1(raw_te) if log_transform else raw_te

    mae_tr  = mean_absolute_error(ytr, pred_tr)
    rmse_tr = mean_squared_error(ytr, pred_tr)**0.5
    r2_tr   = r2_score(ytr, pred_tr)

    mae_te  = mean_absolute_error(yte, pred_te)
    rmse_te = mean_squared_error(yte, pred_te)**0.5
    r2_te   = r2_score(yte, pred_te)

    mask    = yte > 0
    mape_te = np.mean(np.abs((yte[mask] - pred_te[mask]) / yte[mask])) * 100

    results[name] = dict(MAE_train=mae_tr, RMSE_train=rmse_tr, R2_train=r2_tr,
                         MAE_test=mae_te,  RMSE_test=rmse_te,  R2_test=r2_te,
                         MAPE_test=mape_te, best_params=search.best_params_)
    best_models[name] = best

    print(f"  Best params  : {search.best_params_}")
    print(f"  {'':12s} {'MAE':>10s} {'RMSE':>10s} {'R²':>10s}")
    print(f"  {'TRAIN':12s} {mae_tr:>10.3f} {rmse_tr:>10.3f} {r2_tr:>10.3f}")
    print(f"  {'TEST':12s} {mae_te:>10.3f} {rmse_te:>10.3f} {r2_te:>10.3f}")
    print(f"  TEST MAPE    : {mape_te:.2f}%   |   R² gap: {r2_tr-r2_te:.4f}")
    print(f"{'='*62}")
    return best


In [58]:
import xgboost as xgb

# ===================================================================
# MODEL: XGBoost Regressor — Default Parameters
# ===================================================================

xgb_default_model = xgb.XGBRegressor(random_state=42, n_jobs=-1, verbosity=0, objective='reg:squarederror')
xgb_default_model.fit(X_train, y_train)

pred_tr = xgb_default_model.predict(X_train)
pred_te = xgb_default_model.predict(X_test)

mae_tr, rmse_tr, r2_tr = mean_absolute_error(y_train, pred_tr), mean_squared_error(y_train, pred_tr) ** 0.5, r2_score(y_train, pred_tr)
mae_te, rmse_te, r2_te = mean_absolute_error(y_test, pred_te), mean_squared_error(y_test, pred_te) ** 0.5, r2_score(y_test, pred_te)
mask = y_test > 0
mape_te = np.mean(np.abs((y_test[mask] - pred_te[mask]) / y_test[mask])) * 100

results['XGBoost_Default'] = dict(
    MAE_train=mae_tr, RMSE_train=rmse_tr, R2_train=r2_tr,
    MAE_test=mae_te,  RMSE_test=rmse_te,  R2_test=r2_te,
    MAPE_test=mape_te, best_params=xgb_default_model.get_params(),
)
best_models['XGBoost_Default'] = xgb_default_model

print(f"Train → MAE={mae_tr:.3f}  RMSE={rmse_tr:.3f}  R²={r2_tr:.4f}")
print(f"Test  → MAE={mae_te:.3f}  RMSE={rmse_te:.3f}  R²={r2_te:.4f}")


Train → MAE=4.534  RMSE=5.974  R²=0.9659
Test  → MAE=9.472  RMSE=14.245  R²=0.7959


### XGBoost — Hyperparameter Tuning (GridSearchCV)


In [59]:
xgb_param_grid = {
    'n_estimators':     [50],  
    'learning_rate':    [0.09], 
    'max_depth':        [4],   
    'subsample':        [0.79], 
    'colsample_bytree': [0.5],  
    'min_child_weight': [1],  
    'reg_alpha':        [2],  
    'reg_lambda':       [5], 
}

CV_FOLDS = 10   # number of GridSearchCV folds

best_xgb = run_search(
    name='XGBoost_Tuned',
    estimator=xgb.XGBRegressor(random_state=42, n_jobs=-1, verbosity=0, objective='reg:squarederror'),
    param_grid=xgb_param_grid,
    Xtr=X_train, Xte=X_test,
    ytr=y_train, yte=y_test,
    log_transform=False,
)



  Model        : XGBoost_Tuned
  Combinations : 1   |   Folds: 10   |   Total fits: 10
  Best params  : {'colsample_bytree': 0.5, 'learning_rate': 0.09, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 50, 'reg_alpha': 2, 'reg_lambda': 5, 'subsample': 0.79}
                      MAE       RMSE         R²
  TRAIN             8.486     12.180      0.858
  TEST              8.728     13.165      0.826
  TEST MAPE    : 92.26%   |   R² gap: 0.0325


## 9. Save Best Model with joblib


In [60]:
import joblib, os

os.makedirs("models", exist_ok=True)

best_key       = 'XGBoost_Tuned'
best_model_obj = best_models[best_key]

model_path = f"models/target2_{best_key}_best.joblib"
joblib.dump(best_model_obj, model_path)

print(f"Best model: {best_key}")
print(f"Saved to: {model_path}")


Best model: XGBoost_Tuned
Saved to: models/target2_XGBoost_Tuned_best.joblib


## 10. 7-Day Recursive Forecast (by Region)


In [61]:
# ── Static per-region lookups (frozen at historical values) ───────────────────
region_static = feat.groupby('region')[
    ['region_mean', 'region_std', 'region_median', 'region_max', 'region_min', 'region_cat']
].first()

region_dow_static    = feat.groupby(['region', 'dayofweek'])['region_dow_mean'].mean()
region_month_static  = feat.groupby(['region', 'month'])['region_month_mean'].mean()
region_season_static = feat.groupby(['region', 'season'])['region_season_mean'].mean()
region_week_static   = feat.groupby(['region', 'weekofyear'])['region_week_mean'].mean()

hour_pattern_cols_expanded = []
for col in HOUR_PATTERN_COLS:
    hour_pattern_cols_expanded += [f'{col}_lag1', f'{col}_roll7', f'{col}_roll14']
hour_pattern_static = feat.groupby('region')[hour_pattern_cols_expanded].mean()

market_share_static = feat.groupby('region')[['market_share_lag1', 'market_share_roll7']].mean()

order_count_static = feat.groupby('region')[
    ['order_count_lag1', 'order_count_lag7', 'order_count_lag28', 'order_count_roll7']
].mean()

total_daily_series = daily.groupby('date')[TARGET].sum().sort_index()

global_min_date = daily['date'].min()


def recursive_forecast_7day_t2(model, region, future_dates):
    reg_hist = daily[daily['region'] == region].sort_values('date')
    series = reg_hist.set_index('date')[TARGET].astype(float).copy()

    def get_value(s, date):
        if date in s.index:
            return float(s[date])
        if len(s) == 0:
            return 0.0
        return float(s.iloc[-1])

    r_static  = region_static.loc[region]
    hp_static = hour_pattern_static.loc[region]
    ms_static = market_share_static.loc[region]
    oc_static = order_count_static.loc[region]

    preds = []
    for fdate in future_dates:
        row = {}

        # date features
        row['year']  = fdate.year
        row['month'] = fdate.month
        row['day']   = fdate.day
        row['dayofweek']  = fdate.dayofweek
        row['dayofyear']  = fdate.dayofyear
        row['quarter']    = fdate.quarter
        row['weekofyear'] = int(fdate.isocalendar().week)
        row['days_in_month'] = fdate.days_in_month
        row['remaining_days_in_month'] = fdate.days_in_month - fdate.day
        row['is_weekend'] = int(fdate.dayofweek >= 5)
        row['is_weekday'] = int(fdate.dayofweek < 5)
        row['is_month_start'] = int(fdate.is_month_start)
        row['is_month_end']   = int(fdate.is_month_end)
        row['is_quarter_start'] = int(fdate.is_quarter_start)
        row['is_quarter_end']   = int(fdate.is_quarter_end)
        row['is_first_week'] = int(fdate.day <= 7)
        row['is_last_week']  = int(row['remaining_days_in_month'] <= 7)
        row['is_mid_month']  = int(13 <= fdate.day <= 17)
        season_map = {12:0,1:0,2:0,3:1,4:1,5:1,6:2,7:2,8:2,9:3,10:3,11:3}
        row['season'] = season_map[fdate.month]
        row['is_summer'] = int(row['season'] == 2)
        row['is_winter'] = int(row['season'] == 0)
        row['trend'] = (fdate - global_min_date).days

        row['dow_sin']   = np.sin(2*np.pi*row['dayofweek']/7)
        row['dow_cos']   = np.cos(2*np.pi*row['dayofweek']/7)
        row['month_sin'] = np.sin(2*np.pi*row['month']/12)
        row['month_cos'] = np.cos(2*np.pi*row['month']/12)
        row['week_sin']  = np.sin(2*np.pi*row['weekofyear']/52)
        row['week_cos']  = np.cos(2*np.pi*row['weekofyear']/52)
        row['doy_sin']   = np.sin(2*np.pi*row['dayofyear']/366)
        row['doy_cos']   = np.cos(2*np.pi*row['dayofyear']/366)

        # holiday features
        row['is_holiday']             = int(fdate in hol_set)
        row['days_to_holiday']        = days_to_holiday(fdate)
        row['days_to_next_holiday']   = days_to_next_holiday(fdate)
        row['days_from_last_holiday'] = days_from_last_holiday(fdate)
        row['is_holiday_eve']    = int((fdate + pd.Timedelta(days=1)) in hol_set)
        row['is_holiday_after']  = int((fdate - pd.Timedelta(days=1)) in hol_set)
        row['is_long_weekend']   = int(
            fdate.dayofweek in (4,5,6) and
            (row['is_holiday'] or row['is_holiday_eve'] or row['is_holiday_after'])
        )
        row['holiday_week'] = int(row['days_to_holiday'] <= 3)

        # weather — historical monthly average for this region
        sw = weather[(weather['date'].dt.month == fdate.month) & (weather['region'] == region)]
        row['temperature'] = float(sw['temperature'].mean()) if len(sw) else 15.0
        row['rainfall']    = float(sw['rainfall'].mean())    if len(sw) else 2.0
        row['wind_speed']  = float(sw['wind_speed'].mean())  if len(sw) else 15.0
        row['temp_min']    = float(sw['temperature'].min())  if len(sw) else row['temperature'] - 3
        row['temp_max']    = float(sw['temperature'].max())  if len(sw) else row['temperature'] + 3
        row['temp_range']  = row['temp_max'] - row['temp_min']
        row['feels_cold']  = int(row['temperature'] < 5)
        row['feels_hot']   = int(row['temperature'] > 35)
        row['heavy_rain']  = int(row['rainfall'] > 10)
        row['strong_wind'] = int(row['wind_speed'] > 40)

        # hour pattern (frozen historical averages)
        for c in hour_pattern_cols_expanded:
            row[c] = float(hp_static[c])

        # target lags — from the live (recursive) series
        for lag in [1, 2, 3, 7, 14, 21, 28, 364, 365, 366]:
            row[f'lag_{lag}'] = get_value(series, fdate - pd.Timedelta(days=lag))

        row['same_dow_last_week']  = get_value(series, fdate - pd.Timedelta(weeks=1))
        row['same_dow_2weeks_ago'] = get_value(series, fdate - pd.Timedelta(weeks=2))
        row['same_dow_3weeks_ago'] = get_value(series, fdate - pd.Timedelta(weeks=3))
        row['same_dow_4weeks_ago'] = get_value(series, fdate - pd.Timedelta(weeks=4))

        # order_count lags (frozen historical averages — order_count not separately forecast)
        row['order_count_lag1']  = float(oc_static['order_count_lag1'])
        row['order_count_lag7']  = float(oc_static['order_count_lag7'])
        row['order_count_lag28'] = float(oc_static['order_count_lag28'])
        row['order_count_roll7'] = float(oc_static['order_count_roll7'])

        # rolling / ewm features from the live series
        for win in [3, 7, 14, 28]:
            tail = series.iloc[-win:] if len(series) >= win else series
            row[f'roll_mean_{win}']   = float(tail.mean())   if len(tail) else 0.0
            row[f'roll_std_{win}']    = float(tail.std())    if len(tail) > 1 else 0.0
            row[f'roll_max_{win}']    = float(tail.max())    if len(tail) else 0.0
            row[f'roll_min_{win}']    = float(tail.min())    if len(tail) else 0.0
            row[f'roll_median_{win}'] = float(tail.median()) if len(tail) else 0.0

        tail90  = series.iloc[-90:]  if len(series) >= 90  else series
        tail365 = series.iloc[-365:] if len(series) >= 365 else series
        row['roll_mean_90']  = float(tail90.mean())  if len(tail90) else 0.0
        row['roll_mean_365'] = float(tail365.mean()) if len(tail365) else 0.0

        row['ewm_span7']  = float(series.ewm(span=7,  adjust=False).mean().iloc[-1]) if len(series) else 0.0
        row['ewm_span14'] = float(series.ewm(span=14, adjust=False).mean().iloc[-1]) if len(series) else 0.0
        row['ewm_span28'] = float(series.ewm(span=28, adjust=False).mean().iloc[-1]) if len(series) else 0.0

        # momentum
        row['lag_diff_1_7']  = row['lag_1'] - row['lag_7']
        row['lag_diff_7_14'] = row['lag_7'] - row['lag_14']
        row['lag_diff_7_28'] = row['lag_7'] - row['lag_28']
        row['lag_ratio_1_7']  = row['lag_1'] / (row['lag_7']  + 1)
        row['lag_ratio_7_28'] = row['lag_7'] / (row['lag_28'] + 1)
        row['yoy_ratio'] = row['lag_7'] / (row['lag_365'] + 1)
        row['wow_ratio'] = row['roll_mean_7'] / (row['roll_mean_28'] + 1)
        row['roll_range_7']  = row['roll_max_7']  - row['roll_min_7']
        row['roll_range_28'] = row['roll_max_28'] - row['roll_min_28']
        row['mom_growth'] = (row['roll_mean_7'] - row['roll_mean_28']) / (row['roll_mean_28'] + 1)

        # market share (frozen historical averages)
        row['market_share_lag1']  = float(ms_static['market_share_lag1'])
        row['market_share_roll7'] = float(ms_static['market_share_roll7'])

        # total demand across all regions, lagged 1 day — frozen at the last known
        # historical total (future cross-region totals aren't available recursively)
        row['total_daily_lag1'] = get_value(total_daily_series, fdate - pd.Timedelta(days=1))

        # region statistics (static)
        row['region_mean']   = float(r_static['region_mean'])
        row['region_std']    = float(r_static['region_std'])
        row['region_median'] = float(r_static['region_median'])
        row['region_max']    = float(r_static['region_max'])
        row['region_min']    = float(r_static['region_min'])
        row['region_dow_mean']    = float(region_dow_static.get((region, row['dayofweek']), r_static['region_mean']))
        row['region_month_mean']  = float(region_month_static.get((region, row['month']), r_static['region_mean']))
        row['region_season_mean'] = float(region_season_static.get((region, row['season']), r_static['region_mean']))
        row['region_week_mean']   = float(region_week_static.get((region, row['weekofyear']), r_static['region_mean']))

        row['lag1_vs_region_mean']  = row['lag_1'] / (row['region_mean'] + 1)
        row['roll7_vs_region_mean'] = row['roll_mean_7'] / (row['region_mean'] + 1)
        row['roll7_vs_dow_mean']    = row['roll_mean_7'] / (row['region_dow_mean'] + 1)

        row['region_cat'] = int(r_static['region_cat'])

        X_row = pd.DataFrame([row])[feature_cols]
        yhat = float(max(0.0, model.predict(X_row)[0]))
        preds.append(yhat)

        series.loc[fdate] = yhat

    return preds

Run the 7-day recursive forecast for every region:


In [62]:
last_date_t2   = daily['date'].max()
future_dates_t2 = pd.date_range(last_date_t2 + pd.Timedelta(days=1), periods=7, freq='D')

print(f"Generating 7-day recursive forecast with model: {best_key}")
print(f"Forecast window: {future_dates_t2[0].date()} to {future_dates_t2[-1].date()}\n")

forecast_records_t2 = []
for region in regions:
    day_preds = recursive_forecast_7day_t2(best_model_obj, region, future_dates_t2)
    for fdate, pred in zip(future_dates_t2, day_preds):
        forecast_records_t2.append({
            'Region': region,
            'Date': fdate.date(),
            'Day of Week': ['Mon','Tue','Wed','Thu','Fri','Sat','Sun'][fdate.dayofweek],
            'Forecast (item_count_sum)': int(round(pred)),
        })

forecast_df_t2 = pd.DataFrame(forecast_records_t2)
pivot_t2 = forecast_df_t2.pivot(index='Region', columns='Date', values='Forecast (item_count_sum)')
pivot_t2['TOTAL (7 days)'] = pivot_t2.sum(axis=1)
pivot_t2 = pivot_t2.sort_values('TOTAL (7 days)', ascending=False)

print("=" * 80)
print(f"TARGET 2  --  {future_dates_t2[0].date()} to {future_dates_t2[-1].date()}  (7-Day Recursive Forecast)")
print(f"Model: {best_key}  |  Test MAE: {results[best_key]['MAE_test']:.3f}  |  Test R2: {results[best_key]['R2_test']:.4f}")
print("=" * 80)
print(pivot_t2.to_string())
print("=" * 80)
print(f"Total forecast (all regions, 7 days): {pivot_t2['TOTAL (7 days)'].sum():,.0f} item_count")

Generating 7-day recursive forecast with model: XGBoost_Tuned
Forecast window: 2026-06-13 to 2026-06-19

TARGET 2  --  2026-06-13 to 2026-06-19  (7-Day Recursive Forecast)
Model: XGBoost_Tuned  |  Test MAE: 8.728  |  Test R2: 0.8257
Date        2026-06-13  2026-06-14  2026-06-15  2026-06-16  2026-06-17  2026-06-18  2026-06-19  TOTAL (7 days)
Region                                                                                                        
Absheron           121         142         167         140         128         118         104             920
Ganja               36          40          49          38          34          33          29             259
Nakhchivan          35          42          48          37          33          32          27             254
Sheki               19          21          26          19          18          17          15             135
Khachmaz            15          17          20          15          14          14          12       

### 7-day forecast — JSON output

In [63]:
import json

json_records_t2 = []
for region in regions:
    reg_rows = forecast_df_t2[forecast_df_t2['Region'] == region].sort_values('Date')
    json_records_t2.append({
        "region": region,
        "model": best_key,
        "forecast": [
            {"date": str(row['Date']), "day_of_week": row['Day of Week'], "item_count_sum": int(row['Forecast (item_count_sum)'])}
            for _, row in reg_rows.iterrows()
        ],
        "total_7day_item_count": int(reg_rows['Forecast (item_count_sum)'].sum()),
    })

forecast_json_t2 = json.dumps(json_records_t2, indent=2, ensure_ascii=False)

with open("target2_7day_forecast.json", "w", encoding="utf-8") as f:
    f.write(forecast_json_t2)

print(f"Saved 7-day forecast for {len(json_records_t2)} regions to target2_7day_forecast.json")
print(forecast_json_t2[:800])

Saved 7-day forecast for 10 regions to target2_7day_forecast.json
[
  {
    "region": "Absheron",
    "model": "XGBoost_Tuned",
    "forecast": [
      {
        "date": "2026-06-13",
        "day_of_week": "Sat",
        "item_count_sum": 121
      },
      {
        "date": "2026-06-14",
        "day_of_week": "Sun",
        "item_count_sum": 142
      },
      {
        "date": "2026-06-15",
        "day_of_week": "Mon",
        "item_count_sum": 167
      },
      {
        "date": "2026-06-16",
        "day_of_week": "Tue",
        "item_count_sum": 140
      },
      {
        "date": "2026-06-17",
        "day_of_week": "Wed",
        "item_count_sum": 128
      },
      {
        "date": "2026-06-18",
        "day_of_week": "Thu",
        "item_count_sum": 118
      },
      {
        "date": "2026-06-19",
        "day_of_week": "Fri",
        "i
